In [ ]:
import json

from pydantic import BaseModel
import numpy as np

In [ ]:
class Evaluation(BaseModel):
    TEDS: float
    TLAG: float
    rd: float

class Metric(BaseModel):
    mean: float
    median: float
    std: float
    p10: float
    p25: float
    p75: float
    p90: float
    perfect_count: int

class BenchmarkStats(BaseModel):
    benchmark_name: str
    provider_name: str
    TEDS: Metric
    TLAG: Metric
    rd: Metric
    num_evaluations: int
    evaluations: list[Evaluation]

In [ ]:
benchmark_name = "PulseBench-Tab"
provider_name = "pp"

In [ ]:
with open(f"../benchmarks/{benchmark_name}/{provider_name}_filtered_teds.json") as file:
    teds_data = json.load(file)

In [ ]:
with open(f"../benchmarks/{benchmark_name}/{provider_name}_filtered_tlag.json") as file:
    tlag_data = json.load(file)

In [ ]:
with open(f"../benchmarks/{benchmark_name}/{provider_name}_filtered_rd.json") as file:
    rd_data = json.load(file)

In [ ]:
evaluations: list[Evaluation] = []

for teds, tlag, rd in zip(teds_data["per_sample"].values(), tlag_data["per_sample"].values(), rd_data["per_sample"].values()):
    evaluations.append(Evaluation(TEDS=teds, TLAG=tlag, rd=rd))

In [ ]:
def build_metric(data: dict) -> Metric:
    return Metric(
        mean=data["summary"]["mean"],
        median=data["summary"]["median"],
        std=data["summary"]["std"],
        p10=data["summary"]["p10"],
        p25=data["summary"]["p25"],
        p75=data["summary"]["p75"],
        p90=data["summary"]["p90"],
        perfect_count=data["summary"]["perfect_count"],
    )

In [ ]:
stats = BenchmarkStats(
    benchmark_name=benchmark_name,
    provider_name=provider_name,
    TEDS=build_metric(teds_data),
    TLAG=build_metric(tlag_data),
    rd=build_metric(rd_data),
    num_evaluations=len(evaluations),
    evaluations=evaluations,
)

In [ ]:
stats_json = stats.model_dump_json(indent=2)

In [ ]:
with open(f"{benchmark_name}_{provider_name}.json", "w") as file:
    file.write(stats_json)